# AI-Powered Gen Z Credit Card Product Innovation System
## Notebook 05 – AI Product Evaluation Engine

Description

This notebook independently evaluates AI-generated credit card concepts using analytical models developed throughout the project. Each generated product undergoes feature engineering, similarity analysis against the Product Intelligence Database, and product identity classification using the trained Random Forest model. The resulting analytical evidence is then interpreted by an LLM to produce an explainable product evaluation and recommendation for banking product managers.

Main Tasks
• Product feature engineering
• Similarity analysis
• Product identity classification
• Build analytical evidence package
• LLM evidence-based evaluation
• Final recommendation generation

Output
• Product evaluation report
• Similarity results
• Predicted product identity
• AI evaluation
• Final recommendation

# 1. Similarity analysis

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Load the generated concepts
generated_df = pd.read_csv(
    "generated_credit_card_concepts.csv"
)

print("Generated products:", generated_df.shape[0])
print("Available columns:", generated_df.shape[1])

generated_df[
    [
        "product_name",
        "product_category",
        "annual_fee_cad",
        "reward_program_type"
    ]
]

In [ ]:
#Define feature groups
feature_groups = {

    "travel_feature_level": [
        "travel_rewards",
        "airport_lounge_access",
        "travel_insurance",
        "trip_cancellation_insurance",
        "baggage_insurance",
        "rental_car_insurance",
        "no_foreign_transaction_fee",
        "hotel_rewards",
        "airline_rewards"
    ],

    "cashback_feature_level": [
        "cashback_rewards",
        "grocery_rewards",
        "dining_rewards",
        "fuel_rewards",
        "transit_rewards",
        "shopping_rewards"
    ],

    "digital_feature_level": [
        "mobile_wallet_support",
        "apple_pay_support",
        "google_pay_support",
        "samsung_pay_support",
        "virtual_card_support",
        "instant_card_issuance",
        "digital_first_card"
    ],

    "subscription_feature_level": [
        "subscription_rewards",
        "streaming_rewards",
        "netflix_benefit",
        "spotify_benefit",
        "disney_plus_benefit",
        "amazon_prime_benefit",
        "youtube_premium_benefit",
        "food_delivery_benefit",
        "uber_benefit",
        "instacart_benefit",
        "doordash_benefit",
        "skip_benefit"
    ],

    "lifestyle_feature_level": [
        "grocery_rewards",
        "dining_rewards",
        "fuel_rewards",
        "transit_rewards",
        "ride_share_rewards",
        "gaming_rewards",
        "entertainment_rewards",
        "shopping_rewards",
        "food_delivery_benefit"
    ],

    "sustainability_feature_level": [
        "sustainability_rewards",
        "carbon_tracking",
        "green_rewards_program",
        "sustainable_purchase_rewards"
    ],

    "financial_wellness_feature_level": [
        "credit_score_monitoring",
        "budgeting_tools",
        "spending_insights",
        "installment_payment_option",
        "debt_management_support"
    ],

    "security_protection_feature_level": [
        "fraud_alerts",
        "identity_protection",
        "purchase_protection",
        "zero_liability"
    ],

    "premium_benefits_feature_level": [
        "airport_lounge_access",
        "concierge_service",
        "premium_hotel_benefits",
        "luxury_travel_credits",
        "exclusive_event_access"
    ]
}

In [ ]:
#Validate binary features
binary_columns = sorted({
    column
    for source_columns in feature_groups.values()
    for column in source_columns
})

missing_columns = [
    column
    for column in binary_columns
    if column not in generated_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing binary feature columns: {missing_columns}"
    )

invalid_values = {}

for column in binary_columns:

    generated_df[column] = pd.to_numeric(
        generated_df[column],
        errors="coerce"
    )

    invalid = generated_df.loc[
        ~generated_df[column].isin([0, 1]),
        column
    ].unique()

    if len(invalid) > 0:
        invalid_values[column] = invalid.tolist()

if invalid_values:
    raise ValueError(
        f"Invalid binary feature values: {invalid_values}"
    )

print("All required binary features are valid.")

In [ ]:
# Calculate engineered feature levels
for output_column, source_columns in feature_groups.items():

    generated_df[output_column] = (
        generated_df[source_columns]
        .mean(axis=1)
        .mul(100)
        .round(2)
    )

engineered_feature_columns = list(feature_groups.keys())

generated_df[
    ["product_name"] + engineered_feature_columns
]

In [ ]:
#save engineered concepts
generated_df.to_csv(
    "generated_credit_card_concepts_engineered.csv",
    index=False
)

print("Engineered concepts saved.")

In [ ]:
#Load the product intelligence database
product_database = pd.read_csv("synthetic_canadian_consumer_credit_card_product_intelligence_v3.csv"
)

print("Product database shape:", product_database.shape)
product_database.head()

In [ ]:
if "card_category" in product_database.columns:

    product_database = product_database[
        product_database["card_category"]
        .astype(str)
        .str.lower()
        .ne("business")
    ].copy()

print("Products after removing Business cards:", len(product_database))

In [ ]:
missing_database_levels = [
    column
    for column in engineered_feature_columns
    if column not in product_database.columns
]

if missing_database_levels:
    print(
        "The database does not contain these engineered levels:",
        missing_database_levels
    )

    for output_column, source_columns in feature_groups.items():

        missing_sources = [
            column
            for column in source_columns
            if column not in product_database.columns
        ]

        if missing_sources:
            raise ValueError(
                f"Cannot calculate {output_column}. "
                f"Missing database columns: {missing_sources}"
            )

        product_database[output_column] = (
            product_database[source_columns]
            .mean(axis=1)
            .mul(100)
            .round(2)
        )

print("Product database feature levels are ready.")

In [ ]:
similarity_features = [
    "annual_fee_cad",
    "minimum_income_requirement",
    "minimum_credit_score",
    "purchase_interest_rate",
    "foreign_transaction_fee_percent",
    "base_reward_rate",
    "welcome_bonus_value_cad",
    "travel_feature_level",
    "cashback_feature_level",
    "digital_feature_level",
    "subscription_feature_level",
    "lifestyle_feature_level",
    "sustainability_feature_level",
    "financial_wellness_feature_level",
    "security_protection_feature_level",
    "premium_benefits_feature_level"
]

In [ ]:
missing_generated = [
    column
    for column in similarity_features
    if column not in generated_df.columns
]

missing_database = [
    column
    for column in similarity_features
    if column not in product_database.columns
]

if missing_generated:
    raise ValueError(
        f"Missing generated-product similarity columns: "
        f"{missing_generated}"
    )

if missing_database:
    raise ValueError(
        f"Missing database similarity columns: "
        f"{missing_database}"
    )

print("All similarity features are available.")

In [ ]:
for column in similarity_features:

    generated_df[column] = pd.to_numeric(
        generated_df[column],
        errors="coerce"
    )

    product_database[column] = pd.to_numeric(
        product_database[column],
        errors="coerce"
    )

database_medians = product_database[
    similarity_features
].median()

product_database[similarity_features] = (
    product_database[similarity_features]
    .fillna(database_medians)
)

generated_df[similarity_features] = (
    generated_df[similarity_features]
    .fillna(database_medians)
)

In [ ]:
database_mean = product_database[
    similarity_features
].mean()

database_std = product_database[
    similarity_features
].std().replace(0, 1)

database_scaled = (
    product_database[similarity_features] - database_mean
) / database_std

generated_scaled = (
    generated_df[similarity_features] - database_mean
) / database_std

In [ ]:
def find_column(
    dataframe,
    possible_names
):
    for column in possible_names:
        if column in dataframe.columns:
            return column

    return None


product_name_column = find_column(
    product_database,
    [
        "card_name",
        "product_name",
        "card_product_name"
    ]
)

product_category_column = find_column(
    product_database,
    [
        "card_category",
        "product_category"
    ]
)

product_cluster_column = find_column(
    product_database,
    [
        "product_cluster",
        "cluster"
    ]
)

print("Product name column:", product_name_column)
print("Product category column:", product_category_column)
print("Product cluster column:", product_cluster_column)

In [ ]:
similarity_matrix = cosine_similarity(
    generated_scaled,
    database_scaled
)

similarity_results = []

for generated_index, generated_product in generated_df.iterrows():

    similarities = similarity_matrix[generated_index]

    best_database_position = int(
        np.argmax(similarities)
    )

    highest_similarity = float(
        similarities[best_database_position]
    )

    closest_product = product_database.iloc[
        best_database_position
    ]

    result = {
        "product_name": generated_product["product_name"],
        "closest_existing_card": (
            closest_product[product_name_column]
            if product_name_column
            else f"Database product {best_database_position}"
        ),
        "closest_existing_category": (
            closest_product[product_category_column]
            if product_category_column
            else None
        ),
        "closest_existing_cluster": (
            closest_product[product_cluster_column]
            if product_cluster_column
            else None
        ),
        "highest_similarity": round(
            highest_similarity,
            4
        )
    }

    similarity_results.append(result)

similarity_df = pd.DataFrame(similarity_results)

similarity_df

In [ ]:
def interpret_similarity(score):

    if score >= 0.90:
        return (
            "Very high similarity: the concept strongly resembles "
            "an existing product in the database."
        )

    if score >= 0.75:
        return (
            "High similarity: the concept shares many characteristics "
            "with an existing product."
        )

    if score >= 0.55:
        return (
            "Moderate similarity: the concept has both familiar and "
            "differentiated characteristics."
        )

    return (
        "Low similarity: the concept appears relatively differentiated "
        "within the product database."
    )


similarity_df["similarity_interpretation"] = (
    similarity_df["highest_similarity"]
    .apply(interpret_similarity)
)

similarity_df

In [ ]:
evaluation_df = generated_df.merge(
    similarity_df,
    on="product_name",
    how="left"
)

evaluation_df[
    [
        "product_name",
        "closest_existing_card",
        "closest_existing_category",
        "closest_existing_cluster",
        "highest_similarity",
        "similarity_interpretation"
    ]
]

# 2. Product Identity Classification

In [ ]:
rf_model = joblib.load(
    "credit_card_classification_model.pkl"
)

label_encoder = joblib.load(
    "product_type_label_encoder.pkl"
)

print("Classification model loaded.")
print("Expected number of features:", rf_model.n_features_in_)

In [ ]:
classification_features = list(
    rf_model.feature_names_in_
)

print(classification_features)

In [ ]:
missing_classification_features = [
    column
    for column in classification_features
    if column not in evaluation_df.columns
]

if missing_classification_features:
    raise ValueError(
        "Generated concepts are missing classification features: "
        f"{missing_classification_features}"
    )

In [ ]:
classification_input = evaluation_df[
    classification_features
].copy()

for column in classification_features:
    classification_input[column] = pd.to_numeric(
        classification_input[column],
        errors="coerce"
    )

classification_input = classification_input.fillna(0)

In [ ]:
encoded_predictions = rf_model.predict(
    classification_input
).astype(int)

predicted_identities = label_encoder.inverse_transform(
    encoded_predictions
)

evaluation_df["predicted_product_identity"] = (
    predicted_identities
)

evaluation_df[
    [
        "product_name",
        "product_category",
        "predicted_product_identity"
    ]
]

In [ ]:
evaluation_df["predicted_class_code"] = encoded_predictions

evaluation_df[
    [
        "product_name",
        "predicted_class_code",
        "predicted_product_identity"
    ]
]

# 3. LLM Evidence-Based Evaluation

In [ ]:
#check required column s:
required_evaluation_columns = [
    "product_name",
    "product_category",
    "concept_theme",
    "annual_fee_cad",
    "minimum_income_requirement",
    "minimum_credit_score",
    "purchase_interest_rate",
    "foreign_transaction_fee_percent",
    "base_reward_rate",
    "reward_program_type",
    "reward_architecture",
    "customer_value_proposition",
    "business_justification",
    "travel_feature_level",
    "cashback_feature_level",
    "digital_feature_level",
    "subscription_feature_level",
    "lifestyle_feature_level",
    "sustainability_feature_level",
    "financial_wellness_feature_level",
    "security_protection_feature_level",
    "premium_benefits_feature_level",
    "closest_existing_card",
    "closest_existing_category",
    "closest_existing_cluster",
    "highest_similarity",
    "similarity_interpretation",
    "predicted_product_identity"
]

missing_columns = [
    column
    for column in required_evaluation_columns
    if column not in evaluation_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing evaluation columns: {missing_columns}"
    )

print("All required evaluation columns are available.")

In [ ]:
evaluation_df["target_persona"] = "Digital Creators"

In [ ]:
evaluation_df[
    [
        "product_name",
        "target_persona",
        "highest_similarity",
        "predicted_product_identity"
    ]
]

In [ ]:
#create the evidance package
import json
import numpy as np
import pandas as pd


engineered_feature_columns = [
    "travel_feature_level",
    "cashback_feature_level",
    "digital_feature_level",
    "subscription_feature_level",
    "lifestyle_feature_level",
    "sustainability_feature_level",
    "financial_wellness_feature_level",
    "security_protection_feature_level",
    "premium_benefits_feature_level"
]


def safe_value(value):
    """
    Convert NumPy and pandas values into JSON-safe Python values.
    """

    if pd.isna(value):
        return None

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    return value


def build_evaluation_evidence(row):
    """
    Build an evidence package for one generated product.
    The LLM will interpret this evidence but will not recalculate it.
    """

    feature_levels = {
        column: safe_value(row[column])
        for column in engineered_feature_columns
    }

    evidence = {
        "target_persona": safe_value(
            row["target_persona"]
        ),

        "generated_product": {
            "product_name": safe_value(
                row["product_name"]
            ),
            "llm_proposed_category": safe_value(
                row["product_category"]
            ),
            "concept_theme": safe_value(
                row["concept_theme"]
            ),
            "annual_fee_cad": safe_value(
                row["annual_fee_cad"]
            ),
            "minimum_income_requirement": safe_value(
                row["minimum_income_requirement"]
            ),
            "minimum_credit_score": safe_value(
                row["minimum_credit_score"]
            ),
            "purchase_interest_rate": safe_value(
                row["purchase_interest_rate"]
            ),
            "foreign_transaction_fee_percent": safe_value(
                row["foreign_transaction_fee_percent"]
            ),
            "base_reward_rate": safe_value(
                row["base_reward_rate"]
            ),
            "reward_program_type": safe_value(
                row["reward_program_type"]
            ),
            "reward_architecture": safe_value(
                row["reward_architecture"]
            ),
            "customer_value_proposition": safe_value(
                row["customer_value_proposition"]
            ),
            "original_business_justification": safe_value(
                row["business_justification"]
            )
        },

        "engineered_feature_levels": feature_levels,

        "similarity_analysis": {
            "closest_existing_card": safe_value(
                row["closest_existing_card"]
            ),
            "closest_existing_category": safe_value(
                row["closest_existing_category"]
            ),
            "closest_existing_cluster": safe_value(
                row["closest_existing_cluster"]
            ),
            "highest_similarity": safe_value(
                row["highest_similarity"]
            ),
            "similarity_interpretation": safe_value(
                row["similarity_interpretation"]
            )
        },

        "product_identity_classification": {
            "predicted_product_identity": safe_value(
                row["predicted_product_identity"]
            )
        }
    }

    return evidence

In [ ]:
#test
sample_evidence = build_evaluation_evidence(
    evaluation_df.iloc[0]
)

print(
    json.dumps(
        sample_evidence,
        indent=2
    )
)

In [ ]:
#Build the evaluation prompt
def build_evaluation_prompt(evidence):

    evidence_text = json.dumps(
        evidence,
        indent=2
    )

    prompt = f"""
You are a senior Canadian banking product evaluator.

Your role is to evaluate an AI-generated consumer credit card concept
using independent analytical evidence produced by the product evaluation
pipeline.

IMPORTANT LIMITATIONS

- Do not redesign the product.
- Do not invent new product features.
- Do not recalculate the engineered feature levels.
- Do not change the similarity score.
- Do not override the product identity classifier.
- Do not claim that the product is profitable.
- Do not claim proven customer demand or adoption.
- Distinguish clearly between evidence, interpretation, and recommendation.
- Use only the evidence supplied below.
- Do not treat the absence of unrelated feature groups as a weakness.
- Evaluate missing features only when they are important to the selected persona,
  stated value proposition, pricing, or product identity.
- A focused product does not need to appeal to every customer segment.

FOCUS DISCIPLINE

- Evaluate the product only against its selected persona, stated value proposition,
  proposed category, pricing, and analytical evidence.
- Do not treat lack of appeal to unrelated customer segments as a weakness.
- Do not criticize the absence of travel, premium, sustainability, or lifestyle features
  unless those features are necessary for the selected persona or stated product concept.
- A narrowly targeted product may still be a strong recommendation.

EVIDENCE

{evidence_text}

EVALUATION TASK

Evaluate the concept across the following areas:

1. Persona Fit
Assess whether the product appears suitable for the selected Gen Z persona.
Use the product design and engineered feature levels as evidence.

2. Feature Evidence
Explain whether the engineered feature levels support the proposed value
proposition and concept theme.

3. Classification Consistency
Compare the LLM-proposed product category with the predicted product identity.
State whether they are consistent, partially consistent, or inconsistent.

4. Market Differentiation
Interpret the closest existing product and similarity score.
Explain whether the product appears differentiated within the product
intelligence database.

5. Strengths
Identify the strongest evidence-supported aspects of the concept.

6. Weaknesses and Risks

Identify only evidence-supported weaknesses, such as:

- High similarity to an existing product
- Weak differentiation
- Persona misalignment
- Classification inconsistency
- Accessibility concerns
- Pricing or eligibility concerns
- Unsupported business claims
- Features that do not support the stated value proposition

Do not identify missing unrelated feature groups as weaknesses.

Do not recommend adding unrelated travel, premium, sustainability,
or lifestyle features merely to broaden general appeal.

7. Final Recommendation
Select one recommendation:
- Proceed
- Revise
- Do Not Proceed

RECOMMENDATION GUIDANCE

Proceed:
The product shows strong persona fit, coherent feature evidence, reasonable
classification consistency, and meaningful differentiation.

Revise:
The product has potential but has weaknesses such as high similarity,
category inconsistency, weak persona alignment, or pricing and eligibility
concerns.

Do Not Proceed:
The concept has major inconsistencies, weak persona fit, or very limited
differentiation that undermines its value.

OUTPUT REQUIREMENTS

Return only valid JSON using exactly this structure:

{{
  "product_name": "string",
  "persona_fit_rating": "Strong",
  "persona_fit_assessment": "string",
  "feature_evidence_assessment": "string",
  "classification_consistency_rating": "Consistent",
  "classification_consistency_assessment": "string",
  "market_differentiation_rating": "Moderate",
  "market_differentiation_assessment": "string",
  "strengths": [
    "string",
    "string"
  ],
  "weaknesses_or_risks": [
    "string",
    "string"
  ],
  "final_recommendation": "Proceed",
  "recommendation_rationale": "string"
}}

ALLOWED VALUES

persona_fit_rating:
- Strong
- Moderate
- Weak

classification_consistency_rating:
- Consistent
- Partially Consistent
- Inconsistent

market_differentiation_rating:
- High
- Moderate
- Low

final_recommendation:
- Proceed
- Revise
- Do Not Proceed
"""

    return prompt.strip()

In [ ]:
# test prompt
sample_evaluation_prompt = build_evaluation_prompt(
    sample_evidence
)

print(sample_evaluation_prompt)

In [ ]:
import os
import getpass
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = getpass.getpass(
    "Enter your OpenAI API key: "
)

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

print("OpenAI client created successfully.")

In [ ]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Reply with only: API connection successful"
)

print(response.output_text)

In [ ]:
import json
def parse_json_response(response_text):
    cleaned_text = response_text.strip()

    if cleaned_text.startswith("```json"):
        cleaned_text = cleaned_text[7:]
    elif cleaned_text.startswith("```"):
        cleaned_text = cleaned_text[3:]

    if cleaned_text.endswith("```"):
        cleaned_text = cleaned_text[:-3]

    return json.loads(cleaned_text.strip())

In [ ]:
def evaluate_product_with_llm(
    evidence,
    client,
    model="gpt-4.1-mini"
):
    prompt = build_evaluation_prompt(evidence)

    response = client.responses.create(
        model=model,
        input=prompt
    )

    return parse_json_response(
        response.output_text
    )

In [ ]:
#Test one product
sample_evidence = build_evaluation_evidence(
    evaluation_df.iloc[0]
)

sample_evaluation = evaluate_product_with_llm(
    evidence=sample_evidence,
    client=client
)

print(
    json.dumps(
        sample_evaluation,
        indent=2
    )
)

In [ ]:
llm_evaluation_results = []

for index, row in evaluation_df.iterrows():

    print(f"Evaluating {row['product_name']}...")

    evidence = build_evaluation_evidence(row)

    result = evaluate_product_with_llm(
        evidence=evidence,
        client=client
    )

    llm_evaluation_results.append(result)

print("Evaluation completed.")

In [ ]:
llm_evaluation_df = pd.DataFrame(
    llm_evaluation_results
)

llm_evaluation_df

In [ ]:
# Merge with analytical results
final_report_df = evaluation_df.merge(
    llm_evaluation_df,
    on="product_name",
    how="left"
)

final_report_df.head()

In [ ]:
# Make lists readable
def list_to_string(x):
    if isinstance(x, list):
        return "; ".join(x)
    return x

final_report_df["strengths"] = final_report_df["strengths"].apply(list_to_string)
final_report_df["weaknesses_or_risks"] = final_report_df["weaknesses_or_risks"].apply(list_to_string)

In [ ]:
#Save the final report
final_report_df.to_csv(
    "AI_Product_Evaluation_Report.csv",
    index=False
)

final_report_df.to_excel(
    "AI_Product_Evaluation_Report.xlsx",
    index=False
)

print("Evaluation report saved successfully.")